In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

# MCS results (10,000 paths, sim_years = 30): median final capital + depletion
# rate with 95% Wilson CI. Medians transcribed from assets/mcs_summary.md,
# depletion rates and CIs from assets/depletion_rate_ci.md (run 2026-08-04).
data = {
    "Aggressive": {
        "Buy & Hold":  {"median": 270635.72, "dr": 33.04, "ci_lo": 32.12, "ci_hi": 33.97},
        "MSM":         {"median":   9308.23, "dr": 48.97, "ci_lo": 47.99, "ci_hi": 49.95},
        "HMM":         {"median":      0.00, "dr": 89.54, "ci_lo": 88.92, "ci_hi": 90.12},
        "HMM-Uni":     {"median":   1657.38, "dr": 49.73, "ci_lo": 48.75, "ci_hi": 50.71},
        "LSTM":        {"median": 149329.38, "dr": 34.75, "ci_lo": 33.82, "ci_hi": 35.69},
        "Transformer": {"median": 290526.23, "dr": 29.75, "ci_lo": 28.86, "ci_hi": 30.65},
    },
    "Low_Capital": {
        "Buy & Hold":  {"median": 390859.94, "dr": 14.06, "ci_lo": 13.39, "ci_hi": 14.76},
        "MSM":         {"median": 193526.66, "dr": 11.82, "ci_lo": 11.20, "ci_hi": 12.47},
        "HMM":         {"median":      0.00, "dr": 51.99, "ci_lo": 51.01, "ci_hi": 52.97},
        "HMM-Uni":     {"median": 188832.71, "dr": 12.46, "ci_lo": 11.83, "ci_hi": 13.12},
        "LSTM":        {"median": 299773.98, "dr":  8.36, "ci_lo":  7.83, "ci_hi":  8.92},
        "Transformer": {"median": 403422.33, "dr":  9.48, "ci_lo":  8.92, "ci_hi": 10.07},
    },
}

COLORS = {
    "Buy & Hold":  "#7f7f7f",   # Benchmark: neutral gray
    "MSM":         "#1f77b4",   # Markov: blue
    "HMM":         "#1f77b4",
    "HMM-Uni":     "#1f77b4",
    "LSTM":        "#d62728",   # Deep learning: red
    "Transformer": "#d62728",
}
MARKERS = {
    "Buy & Hold":  "s",
    "MSM":         "o",
    "HMM":         "D",
    "HMM-Uni":     "P",
    "LSTM":        "^",
    "Transformer": "v",
}
# MSM and HMM-Uni sit almost on top of each other in both scenarios, so their
# labels are pushed apart vertically (HMM-Uni up, MSM down).
LABEL_OFFSET = {
    "Buy & Hold":  (10, -4),
    "MSM":         (10, -14),
    "HMM":         (10, 4),
    "HMM-Uni":     (10, 8),
    "LSTM":        (10, 4),
    "Transformer": (10, -12),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (scenario, models) in zip(axes, data.items()):
    bh_x, bh_y = models["Buy & Hold"]["median"], models["Buy & Hold"]["dr"]

    # Benchmark quadrants
    ax.axhline(bh_y, color="gray", lw=0.8, ls="--", alpha=0.55, zorder=1)
    ax.axvline(bh_x, color="gray", lw=0.8, ls="--", alpha=0.55, zorder=1)

    for name, v in models.items():
        yerr = [[v["dr"] - v["ci_lo"]], [v["ci_hi"] - v["dr"]]]
        ax.errorbar(
            v["median"], v["dr"], yerr=yerr,
            fmt=MARKERS[name], color=COLORS[name],
            markersize=11, markeredgecolor="black", markeredgewidth=0.6,
            elinewidth=1.2, capsize=3, alpha=0.92, zorder=3,
        )
        ax.annotate(
            name, (v["median"], v["dr"]),
            xytext=LABEL_OFFSET[name], textcoords="offset points",
            fontsize=9.5, fontweight="bold", zorder=4,
        )

    ax.set_xlabel("Median final capital after 30 years", fontsize=10.5)
    ax.set_ylabel("Depletion rate (%) — 95% Wilson CI", fontsize=10.5)
    ax.set_title(scenario.replace("_", "-") + " scenario",
                 fontsize=11.5, fontweight="bold")
    ax.grid(True, alpha=0.25, linestyle=":")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1000:.0f}k €"))

    # some padding on the right/top so annotations fit
    xlo, xhi = ax.get_xlim()
    ylo, yhi = ax.get_ylim()
    ax.set_xlim(xlo, xhi + (xhi - xlo) * 0.10)
    ax.set_ylim(max(0, ylo - (yhi - ylo) * 0.05), yhi + (yhi - ylo) * 0.10)

legend_elements = [
    Patch(facecolor="#1f77b4", edgecolor="black",
          label="Markov models (MSM, HMM, HMM-Uni)"),
    Patch(facecolor="#d62728", edgecolor="black", label="DL models (LSTM, Transformer)"),
    Patch(facecolor="#7f7f7f", edgecolor="black", label="Benchmark (60/40 Buy & Hold)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=10)

fig.suptitle(
    "Risk-return positioning of the models in the stress scenarios",
    fontsize=12.5, fontweight="bold", y=1.00,
)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig("../assets/risk_return_positioning.png", dpi=300, bbox_inches="tight")
plt.show()